# `validate_processed_data`

Notebook mirror of [`src/data/validate_processed_data.py`](../../../src/data/validate_processed_data.py).

The original Python file remains the canonical command-line implementation. CLI entry-point blocks are disabled in this notebook copy so running all cells does not start downloads or model training automatically.

In [ ]:
"""Validate the files generated by ``preprocess_dataset.py``."""

from pathlib import Path
from typing import Dict, Iterable, Set

import pandas as pd

try:
    from load_dataset import PROJECT_ROOT
except ImportError:  # pragma: no cover - supports package-style imports too
    from .load_dataset import PROJECT_ROOT


PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SPLITS = ("train", "validation", "test")
REQUIRED_COLUMNS = {
    "id",
    "clean_title",
    "text_char_count",
    "text_word_count",
    "image_url_valid",
    "image_available",
    "2_way_label",
    "3_way_label",
    "6_way_label",
}
LABEL_COLUMNS = ("2_way_label", "3_way_label", "6_way_label")
BOOLEAN_COLUMNS = ("image_url_valid", "image_available")
COMMENT_COLUMNS = {"id", "comment_count", "top_level_comment_count", "max_comment_ups"}


class ValidationError(ValueError):
    """Raised when a processed output violates a pipeline invariant."""


def _read_chunks(path: Path) -> Iterable[pd.DataFrame]:
    if not path.exists():
        raise ValidationError(f"Missing expected file: {path}")
    return pd.read_csv(path, dtype=str, keep_default_na=False, chunksize=100_000)


def _validate_columns(actual: Iterable[str], required: Set[str], path: Path) -> None:
    missing = sorted(required.difference(actual))
    if missing:
        raise ValidationError(f"{path.name} is missing columns: {', '.join(missing)}")


def _validate_boolean_values(frame: pd.DataFrame, columns: Iterable[str], path: Path) -> None:
    allowed = {"true", "false"}
    for column in columns:
        values = set(frame[column].astype(str).str.strip().str.lower())
        invalid = sorted(values.difference(allowed))
        if invalid:
            raise ValidationError(
                f"{path.name}.{column} contains invalid boolean values: {invalid}"
            )


def validate_split(path: Path, known_ids: Set[str]) -> Dict[str, int]:
    """Validate one processed split and add its IDs to ``known_ids``."""

    rows = 0
    missing_labels = {column: 0 for column in LABEL_COLUMNS}
    first_chunk = True

    for chunk in _read_chunks(path):
        if first_chunk:
            _validate_columns(chunk.columns, REQUIRED_COLUMNS, path)
            first_chunk = False

        ids = chunk["id"].astype(str).str.strip()
        if ids.eq("").any():
            raise ValidationError(f"{path.name} contains empty submission IDs")

        duplicate_ids = set(ids[ids.duplicated(keep=False)])
        if duplicate_ids:
            example = sorted(duplicate_ids)[:3]
            raise ValidationError(f"{path.name} contains duplicate IDs: {example}")

        overlap = set(ids).intersection(known_ids)
        if overlap:
            example = sorted(overlap)[:3]
            raise ValidationError(f"{path.name} overlaps another split: {example}")
        known_ids.update(ids)

        for column in LABEL_COLUMNS:
            missing_labels[column] += int(chunk[column].astype(str).str.strip().eq("").sum())

        _validate_boolean_values(chunk, BOOLEAN_COLUMNS, path)
        rows += len(chunk)

    if first_chunk:
        raise ValidationError(f"{path.name} is empty")

    if any(missing_labels.values()):
        raise ValidationError(f"{path.name} contains missing labels: {missing_labels}")

    return {"rows": rows, "unique_ids": rows}


def validate_comment_summary(submission_ids: Set[str]) -> int:
    """Validate that every summarized comment record joins to a submission."""

    path = PROCESSED_DIR / "comment_summary.csv"
    rows = 0
    first_chunk = True

    for chunk in _read_chunks(path):
        if first_chunk:
            _validate_columns(chunk.columns, COMMENT_COLUMNS, path)
            first_chunk = False

        ids = chunk["id"].astype(str).str.strip()
        if ids.eq("").any():
            raise ValidationError("comment_summary.csv contains empty submission IDs")

        unknown_ids = set(ids).difference(submission_ids)
        if unknown_ids:
            example = sorted(unknown_ids)[:3]
            raise ValidationError(
                f"comment_summary.csv contains IDs absent from the official splits: {example}"
            )

        for column in ("comment_count", "top_level_comment_count"):
            values = pd.to_numeric(chunk[column], errors="coerce")
            if values.isna().any() or (values < 0).any():
                raise ValidationError(f"comment_summary.csv has invalid {column} values")

        rows += len(chunk)
        first_chunk = False

    if first_chunk:
        raise ValidationError("comment_summary.csv is empty")
    return rows


def validate_processed_outputs() -> Dict[str, int]:
    """Validate every processed output and return a compact summary."""

    submission_ids: Set[str] = set()
    summary: Dict[str, int] = {}
    for split in SPLITS:
        summary[split] = validate_split(PROCESSED_DIR / f"{split}.csv", submission_ids)["rows"]
    summary["comment_summary"] = validate_comment_summary(submission_ids)
    return summary


if False and __name__ == "__main__":  # Disabled in the notebook copy; use the source script for CLI execution.
    try:
        result = validate_processed_outputs()
    except ValidationError as error:
        raise SystemExit(f"Validation failed: {error}") from error

    print("Processed data validation passed.")
    for name, rows in result.items():
        print(f"  {name}: {rows:,} rows")
